# Исследовательский анализ Hard Hat Workers v10 (`raw_AllClasses`)

**Цель:** проверить полный датасет Object Detection: 7 035 изображений, YOLO-аннотации и классы `head`, `helmet`, `person`. Ноутбук читает исходные файлы, но не меняет изображения, классы или координаты.

В YOLO одна строка — один Bounding Box: `class_id x_center y_center width height`. Координаты нормированы относительно изображения.

Публичная версия ноутбука хранится без outputs. Агрегированные графики сохраняются в `reports/figures/`; выполненная копия и визуализации исходных фотографий — в `.local/`, исключённом из Git.

## 1. Среда выполнения и конфигурация датасета

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from time import perf_counter
import hashlib
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import yaml
from IPython.display import display

STARTED = perf_counter()
WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (WORKDIR, *WORKDIR.parents)
                     if (p / "data" / "raw").is_dir()), None)
assert PROJECT_ROOT is not None, "Не найден корень проекта с data/raw"
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "hard_hat"
YAML_PATH = DATA_ROOT / "data.yaml"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_FIGURES_DIR = PROJECT_ROOT / ".local" / "research_notes" / "figures"
LOCAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

config = yaml.safe_load(YAML_PATH.read_text(encoding="utf-8"))
raw_names = config["names"]
names = ({i: name for i, name in enumerate(raw_names)} if isinstance(raw_names, list)
         else {int(k): value for k, value in raw_names.items()})
assert config["nc"] == len(names), "nc не совпадает с длиной names"
assert set(names) == set(range(config["nc"])), "ID классов не образуют диапазон 0..nc-1"
print("Python:", platform.python_version())
print("Конфигурация:", config)
print("ID классов:", names)
print("Лицензия в метаданных:", config.get("roboflow", {}).get("license"))
print("Это запись поставщика данных, а не независимая проверка прав на каждую фотографию.")

split_keys = {"train": "train", "valid": "val", "test": "test"}
configured_path_rows = []
for split, yaml_key in split_keys.items():
    raw_path = Path(config[yaml_key])
    resolved_path = (YAML_PATH.parent / raw_path).resolve()
    configured_path_rows.append({
        "часть": split,
        "значение в YAML": config[yaml_key],
        "разрешённый путь": str(resolved_path.relative_to(PROJECT_ROOT.parent)),
        "существует": resolved_path.is_dir(),
    })
configured_paths_df = pd.DataFrame(configured_path_rows).set_index("часть")
display(configured_paths_df)

## 2. Структура частей и соответствие файлов

In [ ]:
SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
image_paths, label_paths, structure_rows = {}, {}, []
for split in SPLITS:
    image_dir = DATA_ROOT / split / "images"
    label_dir = DATA_ROOT / split / "labels"
    assert image_dir.is_dir() and label_dir.is_dir(), f"Нет images/labels для {split}"
    images = sorted(p for p in image_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = sorted(label_dir.glob("*.txt"))
    image_paths[split], label_paths[split] = images, labels
    image_stems, label_stems = {p.stem for p in images}, {p.stem for p in labels}
    structure_rows.append({"часть": split, "изображений": len(images), "txt": len(labels),
                           "без txt": len(image_stems - label_stems),
                           "лишних txt": len(label_stems - image_stems)})
structure_df = pd.DataFrame(structure_rows).set_index("часть")
display(structure_df)
print("Всего изображений:", int(structure_df["изображений"].sum()))
print("Всего файлов разметки:", int(structure_df["txt"].sum()))

## 3. Техническая валидация YOLO-аннотаций

Отдельно считаются: неверное число полей, нечисловые значения, нецелый/неизвестный ID, координаты вне `[0,1]`, неположительные размеры и фактический выход рамки за границы. Небольшой допуск не используется: проверяется строгая геометрия.

In [ ]:
annotations, per_image_rows = [], []
issue_counts = Counter()
issue_examples = defaultdict(list)
for split in SPLITS:
    labels_by_stem = {p.stem: p for p in label_paths[split]}
    for image_path in image_paths[split]:
        path = labels_by_stem.get(image_path.stem)
        lines = [] if path is None else path.read_text(encoding="utf-8").splitlines()
        if not any(line.strip() for line in lines):
            issue_counts["пустой файл разметки"] += 1
        counts = Counter()
        for line_number, line in enumerate(lines, start=1):
            if not line.strip():
                issue_counts["пустая строка"] += 1
                continue
            parts = line.split()
            problem = None
            if len(parts) != 5:
                problem = f"не 5 полей ({len(parts)})"
            else:
                try:
                    values = [float(value) for value in parts]
                except ValueError:
                    problem = "нечисловое значение"
            if problem is None:
                class_float, x, y, width, height = values
                if not class_float.is_integer():
                    problem = "ID класса не целый"
                else:
                    class_id = int(class_float)
                    if class_id not in names:
                        problem = "неизвестный ID класса"
                    elif not np.isfinite([x, y, width, height]).all():
                        problem = "NaN или бесконечность"
                    elif not (0 <= x <= 1 and 0 <= y <= 1):
                        problem = "центр вне [0, 1]"
                    elif not (0 < width <= 1 and 0 < height <= 1):
                        problem = "неположительный/слишком большой размер"
                    elif (x - width / 2 < 0 or y - height / 2 < 0 or
                          x + width / 2 > 1 or y + height / 2 > 1):
                        problem = "рамка выходит за изображение"
            if problem is not None:
                issue_counts[problem] += 1
                if len(issue_examples[problem]) < 5:
                    issue_examples[problem].append((split, path.name if path else None,
                                                    line_number, line))
                continue
            annotations.append({"split": split, "stem": image_path.stem,
                                "class_id": class_id, "class": names[class_id],
                                "x_center": x, "y_center": y,
                                "width": width, "height": height,
                                "area_part": width * height})
            counts[class_id] += 1
        per_image_rows.append({"split": split, "stem": image_path.stem,
                               "objects": sum(counts.values()),
                               **{f"count_{names[cid]}": counts[cid] for cid in names}})

ann_df = pd.DataFrame(annotations)
per_image_df = pd.DataFrame(per_image_rows)
print("Корректных Bounding Boxes:", len(ann_df))
print("Нарушения по типам:", dict(issue_counts) if issue_counts else "не обнаружены")
if issue_examples:
    display(pd.DataFrame([(kind, *row) for kind, rows in issue_examples.items() for row in rows],
                         columns=["проблема", "часть", "файл", "строка", "содержимое"]))

## 4. Распределение классов по объектам и изображениям

In [ ]:
objects_by_class = ann_df["class"].value_counts().reindex(names.values(), fill_value=0)
objects_by_split = pd.crosstab(ann_df["split"], ann_df["class"]).reindex(
    index=SPLITS, columns=list(names.values()), fill_value=0)
images_by_class = pd.Series({name: per_image_df[f"count_{name}"].gt(0).sum()
                             for name in names.values()})
images_by_split = pd.DataFrame({
    name: per_image_df.assign(has=per_image_df[f"count_{name}"].gt(0)).groupby("split")["has"].sum()
    for name in names.values()}).reindex(SPLITS)
print("Количество Bounding Boxes:")
display(objects_by_class.rename("bbox").to_frame())
display(objects_by_split)
print("Количество изображений, содержащих класс:")
display(images_by_class.rename("изображений").to_frame())
display(images_by_split)
print(f"person: {objects_by_class['person']} bbox ({objects_by_class['person']/len(ann_df)*100:.2f}%) "
      f"на {images_by_class['person']} изображениях ({images_by_class['person']/len(per_image_df)*100:.2f}%).")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
objects_by_class.plot.bar(ax=axes[0], color=["#E15759", "#59A14F", "#4C78A8"])
axes[0].set(title="Bounding Boxes по классам", xlabel="Класс", ylabel="Количество bbox")
images_by_class.plot.bar(ax=axes[1], color=["#E15759", "#59A14F", "#4C78A8"])
axes[1].set(title="Изображения, содержащие класс", xlabel="Класс", ylabel="Изображений")
for ax in axes: ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hard_hat_class_balance.png", dpi=150)
plt.show()

## 5. Плотность объектов и сочетания классов

In [ ]:
display(per_image_df["objects"].describe(percentiles=[.25, .50, .75, .90, .95, .99]).to_frame())
print("Изображений без корректных объектов:", int(per_image_df["objects"].eq(0).sum()))
print("Максимум объектов:", int(per_image_df["objects"].max()))
display(per_image_df.nlargest(5, "objects")[["split", "stem", "objects"]])

has_head = per_image_df["count_head"].gt(0)
has_helmet = per_image_df["count_helmet"].gt(0)
has_person = per_image_df["count_person"].gt(0)
combinations = pd.Series({
    "head и helmet (включая person)": (has_head & has_helmet).sum(),
    "все три класса": (has_head & has_helmet & has_person).sum(),
    "только head": (has_head & ~has_helmet & ~has_person).sum(),
    "только helmet": (has_helmet & ~has_head & ~has_person).sum(),
    "только person": (has_person & ~has_head & ~has_helmet).sum(),
}).astype(int)
display(combinations.rename("изображений").to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(per_image_df["objects"], bins=np.arange(.5, per_image_df["objects"].max() + 1.5), color="#4C78A8")
ax.set(title="Количество размеченных объектов на изображении", xlabel="Объектов", ylabel="Изображений")
ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hard_hat_objects_per_image.png", dpi=150)
plt.show()

## 6. Форматы, размеры и техническая читаемость изображений

In [ ]:
image_rows, unreadable_images = [], []
for split in SPLITS:
    for path in image_paths[split]:
        try:
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                width, height = image.size
                mode, image_format = image.mode, image.format
            image_rows.append({"split": split, "stem": path.stem, "width": width,
                               "height": height, "aspect": width / height,
                               "mode": mode, "format": image_format})
        except Exception as error:
            unreadable_images.append((split, path.name, repr(error)))
size_df = pd.DataFrame(image_rows)
print("Прочитано:", len(size_df), "| нечитаемых:", len(unreadable_images))
display(size_df[["width", "height", "aspect"]].describe(
    percentiles=[.01, .25, .50, .75, .99]).round(3))
print("Цветовые режимы:", size_df["mode"].value_counts().to_dict())
print("Форматы:", size_df["format"].value_counts().to_dict())
print("Самые частые разрешения:")
display(size_df.groupby(["width", "height"]).size().nlargest(10).rename("изображений").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(size_df["width"], bins=35, color="#4C78A8")
axes[0].set(title="Ширина изображений", xlabel="Пиксели", ylabel="Изображений")
axes[1].hist(size_df["aspect"], bins=35, color="#F28E2B")
axes[1].set(title="Соотношение сторон", xlabel="Ширина / высота", ylabel="Изображений")
for ax in axes: ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hard_hat_image_dimensions.png", dpi=150)
plt.show()

## 7. Геометрические характеристики Bounding Boxes

In [ ]:
box_stats = ann_df.groupby("class")[["width", "height", "area_part"]].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99])
display(box_stats.round(5))
size_thresholds = pd.Series({
    "очень маленькие: площадь < 0,1% кадра": ann_df["area_part"].lt(.001).sum(),
    "маленькие: площадь < 1% кадра": ann_df["area_part"].lt(.01).sum(),
    "крупные: площадь > 25% кадра": ann_df["area_part"].gt(.25).sum(),
    "очень крупные: площадь > 50% кадра": ann_df["area_part"].gt(.50).sum(),
}).astype(int)
display(pd.DataFrame({"bbox": size_thresholds,
                      "% bbox": size_thresholds / len(ann_df) * 100}).round(3))
by_class_thresholds = ann_df.groupby("class")["area_part"].agg(
    bbox="size", lt_0_1pct=lambda s: s.lt(.001).sum(),
    lt_1pct=lambda s: s.lt(.01).sum(), gt_25pct=lambda s: s.gt(.25).sum())
display(by_class_thresholds)

fig, ax = plt.subplots(figsize=(9, 4))
for class_name, color in zip(names.values(), ["#E15759", "#59A14F", "#4C78A8"]):
    values = ann_df.loc[ann_df["class"].eq(class_name), "area_part"].clip(upper=.25)
    ax.hist(values, bins=50, density=True, histtype="step", linewidth=1.8,
            label=class_name, color=color)
ax.set(title="Доля площади кадра у bbox (25%+ объединены)", xlabel="Доля площади", ylabel="Плотность")
ax.legend(); ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hard_hat_box_areas.png", dpi=150)
plt.show()

## 8. Выборочный визуальный аудит разметки

Цвета: `head` — красный, `helmet` — зелёный, `person` — синий. Примеры выбраны по содержимому аннотаций, а не случайно: одиночная каска, много касок, головы без касок, много людей, все классы и смешанная сцена. Фигура сохраняется только в `.local/`, поскольку содержит исходные фотографии с людьми.

In [ ]:
example_specs = [
    ("train", "004697_jpg.rf.a6ffd6e3d6220e445556c95a3188460f", "одна каска"),
    ("train", "004819_jpg.rf.ebbb18a4bd04d48b69b3ebb21e7e92a4", "много касок"),
    ("valid", "006026_jpg.rf.6e7289d466512fc215b0d7e6877a8692", "головы без касок"),
    ("train", "002646_jpg.rf.c9ac6b967a34f0c5fe0541170028717f", "много person"),
    ("train", "002463_jpg.rf.066f52df5491c94fc7d30c47596217c9", "все три класса"),
    ("train", "000338_jpg.rf.46ae2f92840ff15499a769e603c9efab", "сложная смешанная сцена"),
]
colors = {0: "red", 1: "lime", 2: "deepskyblue"}
fig, axes = plt.subplots(3, 2, figsize=(15, 16))
for ax, (split, stem, title) in zip(axes.flat, example_specs):
    image_path = next((p for p in image_paths[split] if p.stem == stem), None)
    assert image_path is not None, f"Не найден пример {stem}"
    with Image.open(image_path) as image:
        rgb = image.convert("RGB")
        width, height = rgb.size
        ax.imshow(rgb)
    label_path = DATA_ROOT / split / "labels" / f"{stem}.txt"
    for line in label_path.read_text(encoding="utf-8").splitlines():
        class_id, x, y, box_width, box_height = map(float, line.split())
        class_id = int(class_id)
        left, top = (x - box_width / 2) * width, (y - box_height / 2) * height
        ax.add_patch(Rectangle((left, top), box_width * width, box_height * height,
                               fill=False, linewidth=1.5, edgecolor=colors[class_id]))
        ax.text(left, max(0, top - 3), names[class_id], color=colors[class_id], fontsize=7,
                bbox={"facecolor": "black", "alpha": .55, "edgecolor": "none", "pad": 1})
    ax.set_title(f"{title} — {split}/{stem[:12]}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(LOCAL_FIGURES_DIR / "hard_hat_annotation_examples.png", dpi=160)
plt.show()

Наблюдения по показанным кадрам:

- в примере «головы без касок» класс `head` действительно охватывает непокрытые головы; это подтверждает предполагаемый смысл **на этом и сходных просмотренных примерах**, но codebook источника всё равно нужен;
- в примере «одна каска» человек виден целиком, однако рамки `person` нет; следовательно, `person` не является исчерпывающей разметкой хотя бы на этом кадре;
- в других сценах `person` размечен вместе с `helmet`. Такая выборочная последовательность и редкость класса требуют отдельного аудита перед трёхклассовым экспериментом;
- рамки визуально совмещаются с объектами на этих шести кадрах, но шесть примеров не доказывают полноту и качество всех 7 035 изображений.

## 9. Проверка точных дубликатов и риска утечки

In [ ]:
hash_groups = defaultdict(list)
for split, paths in image_paths.items():
    for path in paths:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        hash_groups[digest].append((split, path.name))
exact_duplicate_groups = [items for items in hash_groups.values() if len(items) > 1]
cross_split_duplicate_groups = [items for items in exact_duplicate_groups
                                if len({split for split, _ in items}) > 1]
print("Групп точных дубликатов в полном наборе:", len(exact_duplicate_groups))
print("Лишних файлов-копий:", sum(len(items) - 1 for items in exact_duplicate_groups))
print("Групп точных дубликатов между частями:", len(cross_split_duplicate_groups))
if cross_split_duplicate_groups:
    display(pd.DataFrame({"совпадения": cross_split_duplicate_groups}))
print("SHA-256 находит только побайтовые совпадения; кадры после ресайза, перекодирования или небольшого сдвига он не выявляет.")

## 10. Ограничения и рекомендации по подготовке данных

**Вариант A — `head`, `helmet`, `person`:** сохраняет более широкую задачу, но `person` составляет очень малую долю рамок и визуально размечен не на всех кадрах с людьми. Понадобятся аудит codebook, дополнительная разметка `person`, взвешивание классов и стратификация сцен.

**Вариант B — `head`, `helmet`:** лучше соответствует задаче контроля касок, содержит гораздо больше примеров и проще для согласованной двойной разметки. По текущему EDA это более надёжный основной вариант, если преподаватель разрешит сузить постановку. Исходный `person` удалять нельзя.

Для первого быстрого прототипа можно использовать стратифицированную подвыборку около 2 000–3 000 изображений, но для финального обучения предпочтительно сохранить все 7 035: объём умеренный, а мелких объектов много. Новые реальные кадры следует размечать по единому codebook двумя людьми. При синтетических поворотах, обрезке и масштабировании координаты bbox нужно преобразовывать вместе с изображением; синтетику создавать только после фиксации split, чтобы не загрязнить validation/test.

In [ ]:
print("ИТОГОВАЯ СВОДКА")
print("Изображения:", int(structure_df["изображений"].sum()), structure_df["изображений"].to_dict())
print("Bounding Boxes:", len(ann_df), objects_by_class.to_dict())
print("Изображения с классами:", images_by_class.to_dict())
print("Пустые изображения/аннотации:", int(per_image_df["objects"].eq(0).sum()))
print("Нарушения аннотаций:", dict(issue_counts))
print("Нечитаемые изображения:", len(unreadable_images))
print("Точные дубликаты между частями:", len(cross_split_duplicate_groups))
print("Пути split из исходного YAML существуют:", bool(configured_paths_df["существует"].all()))
print(f"Время выполнения внутри ноутбука: {perf_counter() - STARTED:.2f} с")